# GPU inference smoke test
    ## Objective
    Load the exported state dictionary without retraining or downloading pretrained weights, and score one authorized user-supplied local image.
    Softmax values are uncalibrated model scores, not emotional measurements.

In [ ]:
import sys
import torch
import torchvision
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU-ONLY TRAINING: CUDA unavailable in this kernel. Fix NVIDIA driver, CUDA wheel, interpreter or notebook kernel. CPU training is forbidden.")
device = torch.device("cuda:0")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU properties:", torch.cuda.get_device_properties(0))
x = torch.randn((128, 128), device=device)
y = x @ x
assert y.is_cuda and y.device.type == "cuda"
torch.cuda.synchronize()
print("GPU tensor test PASSED:", y.device)
del x, y

## Setup and choose a local image
The default smoke input is one image from the user-supplied training partition. This checks checkpoint loading and output shape, not generalization. Set `IMAGE_PATH` to another authorized local file if desired. Do not use held-out test images for iterative model tuning.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open the FaceAttend project folder.")
sys.path.insert(0, str(ROOT / "src"))
os.environ["TORCH_HOME"] = str(ROOT / ".cache/torch")
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
from faceattend.utils import environment, load_config, seed_everything, write_json

In [ ]:
IMAGE_PATH = ROOT / pd.read_csv(ROOT / "data/processed/train.csv").iloc[0].path
# Optional override: IMAGE_PATH = ROOT / "data/local_example.jpg"
CHECKPOINT = ROOT / "models/faceattend_best.pt"
if not CHECKPOINT.exists():
    raise FileNotFoundError("No exported model. Complete both training runs and notebook 04 first.")
if IMAGE_PATH is None or not Path(IMAGE_PATH).is_file():
    raise FileNotFoundError("Set IMAGE_PATH to your authorized local image.")

## Reconstruct matching preprocessing and run on CUDA
Scores follow the fixed seven-class order; repeated inference checks consistent output.

In [ ]:
from faceattend.inference import predict_image
from faceattend import CLASS_NAMES
import numpy as np
scores = predict_image(CHECKPOINT, IMAGE_PATH)
repeat = predict_image(CHECKPOINT, IMAGE_PATH)
assert list(scores) == CLASS_NAMES and len(scores) == 7
assert np.isclose(sum(scores.values()), 1.0)
assert np.allclose(list(scores.values()), list(repeat.values()), atol=1e-6)
display(pd.Series(scores, name="Uncalibrated visible-expression score"))
print("Predicted annotation category:", max(scores, key=scores.get))

## Limitations
    Out-of-distribution faces, occlusion, lighting, pose and ambiguous expressions can cause mistakes.
    No identity matching or attendance action occurs. Scores are not calibrated confidence and must not be used for student/employee assessment.
    Do not publish local face-image outputs.